# Chapter 10: Tokenization — Text to Numbers

Neural networks only understand numbers. Tokenization is the **first step** in any language model:

```
"Hello world" → [15496, 995] → neural network
```

The tokenizer decides **how to split text into pieces** and assigns each piece an ID number.

---
## Three Approaches to Tokenization

| Level | Split by | Example: "unhappiness" | Pros | Cons |
|-------|----------|----------------------|------|------|
| **Word** | spaces/punctuation | `["unhappiness"]` | Simple | Huge vocabulary, can't handle new words |
| **Character** | each letter | `["u","n","h","a","p","p","i","n","e","s","s"]` | Tiny vocabulary | Very long sequences, loses meaning |
| **Subword** | learned pieces | `["un", "happiness"]` | Best of both | Needs training to find the pieces |

In [ ]:
# Compare the three approaches
text = "The unhappiest cat couldn't sleep"

# Word-level
word_tokens = text.split()

# Character-level
char_tokens = list(text)

# Subword (simulated BPE-style)
subword_tokens = ["The", " un", "happ", "iest", " cat", " couldn", "'t", " sleep"]

print(f"Text: '{text}'\n")
print(f"Word-level:      {word_tokens}")
print(f"  → {len(word_tokens)} tokens")
print(f"  Problem: 'couldn't' is one token. 'unhappiest' is one token.")
print(f"  If model never saw 'unhappiest' in training → unknown word!\n")

print(f"Character-level: {char_tokens}")
print(f"  → {len(char_tokens)} tokens")
print(f"  Problem: 32 tokens for 5 words! Sequences become very long.\n")

print(f"Subword (BPE):   {subword_tokens}")
print(f"  → {len(subword_tokens)} tokens")
print(f"  'un' + 'happ' + 'iest' — model understands the parts!")
print(f"  Even new words can be split into known subwords.")

---
## BPE: Byte Pair Encoding

The most common subword tokenization (used by GPT, Claude, etc).

**Training a BPE tokenizer** (done once, before model training):

1. Start with individual characters as vocabulary
2. Count all adjacent pairs in the text
3. Merge the most frequent pair into a new token
4. Repeat until vocabulary reaches desired size

The result: common words become single tokens, rare words get split into pieces.

In [ ]:
# BPE from scratch
import re
from collections import Counter

def get_pairs(tokens_list):
    """Count adjacent pairs across all words."""
    pairs = Counter()
    for word_tokens in tokens_list:
        for i in range(len(word_tokens) - 1):
            pairs[(word_tokens[i], word_tokens[i+1])] += 1
    return pairs

def merge_pair(tokens_list, pair):
    """Merge all occurrences of a pair into a single token."""
    new_list = []
    for word_tokens in tokens_list:
        new_tokens = []
        i = 0
        while i < len(word_tokens):
            if i < len(word_tokens) - 1 and (word_tokens[i], word_tokens[i+1]) == pair:
                new_tokens.append(word_tokens[i] + word_tokens[i+1])
                i += 2
            else:
                new_tokens.append(word_tokens[i])
                i += 1
        new_list.append(new_tokens)
    return new_list

# Training corpus
corpus = "the cat sat on the mat the cat ate the rat"
words = corpus.split()

# Start: each word split into characters + end marker
tokens_list = [list(w) + ["_"] for w in words]

print("BPE Training Step by Step")
print(f"Corpus: '{corpus}'\n")
print(f"Initial (characters): {tokens_list[:4]}...\n")

vocab = set()
for tl in tokens_list:
    vocab.update(tl)
print(f"Initial vocabulary ({len(vocab)}): {sorted(vocab)}\n")
print("─" * 60)

# Run BPE for several merges
for step in range(8):
    pairs = get_pairs(tokens_list)
    if not pairs:
        break
    best_pair = pairs.most_common(1)[0]
    pair, count = best_pair
    
    print(f"\nMerge {step + 1}: '{pair[0]}' + '{pair[1]}' → '{pair[0]+pair[1]}' (appeared {count} times)")
    
    tokens_list = merge_pair(tokens_list, pair)
    vocab.add(pair[0] + pair[1])
    
    # Show current state of first few words
    unique_words = []
    seen = set()
    for w, t in zip(words, tokens_list):
        if w not in seen:
            unique_words.append((w, t))
            seen.add(w)
    for w, t in unique_words[:5]:
        print(f"  '{w}' → {t}")

print(f"\n{'─' * 60}")
print(f"Final vocabulary ({len(vocab)}): {sorted(vocab)}")
print(f"\n'the' became a single token through merges: t→th→the")

---
## How Real Tokenizers Work

Real LLM tokenizers (GPT-4, Claude) use BPE with ~100,000 tokens in the vocabulary.

Common words → 1 token: `"the"` → `[1]`

Rare words → multiple tokens: `"cryptocurrency"` → `["crypt", "ocur", "rency"]`

Numbers and code often get split character by character.

In [ ]:
# Simulate a trained tokenizer
# (Real one would have 100K entries learned from billions of words)

vocab_table = {
    "the": 1, " ": 2, "cat": 3, "sat": 4, "on": 5,
    "mat": 6, "un": 7, "happ": 8, "y": 9, "iest": 10,
    "dog": 11, "s": 12, "ran": 13, "quick": 14, "ly": 15,
    ".": 16, "!": 17, "a": 18, "I": 19, "is": 20,
    "it": 21, "not": 22, "ing": 23, "ed": 24, "er": 25,
    "to": 26, "go": 27, "play": 28, "sleep": 29,
}

def simple_tokenize(text, vocab):
    """Greedy longest-match tokenization."""
    tokens = []
    i = 0
    while i < len(text):
        best = None
        for length in range(min(10, len(text) - i), 0, -1):
            candidate = text[i:i+length]
            if candidate in vocab:
                best = candidate
                break
        if best:
            tokens.append((best, vocab[best]))
            i += len(best)
        else:
            tokens.append((text[i], "?"))
            i += 1
    return tokens

test_texts = [
    "the cat sat on the mat",
    "the unhappiest dog ran quickly",
    "it is not a cat",
]

print("Tokenization Examples:\n")
for text in test_texts:
    tokens = simple_tokenize(text, vocab_table)
    pieces = [t[0] for t in tokens]
    ids = [t[1] for t in tokens]
    print(f"  Text:   '{text}'")
    print(f"  Tokens: {pieces}")
    print(f"  IDs:    {ids}")
    print(f"  Count:  {len(tokens)} tokens")
    print()

---
## Token Count Matters: Why LLMs Have "Context Windows"

LLMs have a maximum number of tokens they can process at once:

```
GPT-3:    4,096 tokens   (~3,000 words)
GPT-4:    128,000 tokens (~96,000 words)
Claude:   200,000 tokens (~150,000 words)
```

More tokens = more memory and computation (attention is O(n²) — Ch 12).

In [ ]:
# Token vs word count
print("Tokens vs Words (approximate ratios):\n")

examples = [
    ("English text", "The quick brown fox jumps over the lazy dog near the river bank", 1.3),
    ("Python code", "def calculate_average(numbers):\n    return sum(numbers) / len(numbers)", 1.8),
    ("JSON data", '{"name": "John", "age": 30, "city": "New York"}', 2.0),
    ("URLs", "https://www.example.com/path/to/resource?param=value", 3.0),
]

print(f"{'Type':<16} {'Words':>6} {'~Tokens':>8} {'Ratio':>6}")
print(f"{'─'*16} {'─'*6} {'─'*8} {'─'*6}")

for name, text, ratio in examples:
    word_count = len(text.split())
    token_est = int(word_count * ratio)
    print(f"{name:<16} {word_count:>6} {token_est:>8} {ratio:>5.1f}x")

print("\nRule of thumb: 1 token ≈ 0.75 words (or 4 characters) for English.")
print("Code and special characters use more tokens per word.")

---
## Special Tokens

Tokenizers include special tokens that the model uses as signals:

```
[BOS]  → Beginning of sequence ("start here")
[EOS]  → End of sequence ("stop generating")
[PAD]  → Padding (fill short sequences to equal length)
[UNK]  → Unknown token (not in vocabulary)
[SEP]  → Separator between segments
[MASK] → Hidden token (for training, like a fill-in-the-blank)
```

In [ ]:
# Demonstrate special tokens and padding
print("Special Tokens in Practice:\n")

# Batch of sentences with different lengths
sentences = [
    ["[BOS]", "the", "cat", "sat", "[EOS]"],
    ["[BOS]", "I", "go", "[EOS]", "[PAD]"],
    ["[BOS]", "it", "ran", "quickly", "[EOS]"],
]

special_ids = {"[BOS]": 0, "[EOS]": 1, "[PAD]": 2, "[UNK]": 3}

print("Batching sentences (must be same length for GPU):")
print()
for i, sent in enumerate(sentences):
    display = "  ".join(f"{t:>8}" for t in sent)
    print(f"  Sent {i+1}: {display}")

print()
print("  [BOS] tells the model: 'this is the start'")
print("  [EOS] tells the model: 'stop generating here'")
print("  [PAD] is ignored — just fills space so all sentences are same length")
print()
print("  The model learns what these special tokens mean during training.")

---
## The Full Pipeline: Text → Tokens → IDs → Model

```
"Hello, how are you?"
    ↓ tokenize
["Hello", ",", " how", " are", " you", "?"]
    ↓ look up IDs
[15496, 11, 703, 527, 499, 30]
    ↓ embedding (Ch 11)
[[0.2, -0.1, 0.8, ...], [0.5, 0.3, ...], ...]
    ↓ transformer (Ch 12)
prediction
```

Tokenization is purely mechanical — no learning. But it deeply affects model performance:
- Too few tokens (word-level): can't handle new words
- Too many tokens (character): sequences too long, slow
- Just right (subword): efficient and flexible

In [ ]:
# Full pipeline demonstration
import numpy as np
np.random.seed(42)

# Step 1: Tokenize
text = "the cat sat"
tokens = text.split()  # simple word tokenization for demo

# Step 2: Token → ID
vocab = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4, "dog": 5}
ids = [vocab[t] for t in tokens]

# Step 3: ID → Embedding (learned vectors, shown in Ch 11)
embedding_dim = 4
embedding_table = np.random.randn(len(vocab), embedding_dim) * 0.3
embeddings = np.array([embedding_table[i] for i in ids])

print("Full Pipeline: Text → Numbers\n")

print(f"Step 1 — Tokenize:")
print(f"  '{text}' → {tokens}\n")

print(f"Step 2 — Token → ID:")
for t, i in zip(tokens, ids):
    print(f"  '{t}' → {i}")

print(f"\nStep 3 — ID → Embedding vector:")
for t, i, emb in zip(tokens, ids, embeddings):
    print(f"  {i} ('{t}') → [{', '.join(f'{v:+.3f}' for v in emb)}]")

print(f"\nStep 4 — These vectors go into the Transformer (Ch 12)")
print(f"\nThe model NEVER sees the text '{text}'.")
print(f"It only sees: {embeddings.round(3).tolist()}")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Tokenization** | Splits text into pieces the model can process |
| **Word-level** | Simple but huge vocab, can't handle new words |
| **Character-level** | Tiny vocab but very long sequences |
| **Subword (BPE)** | Best of both — learned splits, used by modern LLMs |
| **Vocabulary** | Lookup table: token → integer ID |
| **Special tokens** | BOS, EOS, PAD — control signals for the model |
| **Context window** | Max tokens model can process (cost is O(n²)) |

**Next chapter**: Embeddings — how token IDs become meaningful vectors